In [1]:
from peft import LoraConfig, get_peft_model
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from train_gym.rmt.rmt_wrappers import (
    MemoryCell,
    RecurrentWrapper,
    MemoryCellTrain,
    RecurrentWrapperTrain,
    MemoryCellTrainLiger,
    lce_forward,
    ChatMemoryCellTrain,
)

model_path = (
    # "model_checkpoints/sft/rmt_sft_lora_Qwen3-1.7B_WildChat-4.8M_v3/checkpoint-21000"
    "model_checkpoints/sft/rmt_sft_full_Qwen3-1.7B_WildChat-4.8M_v1/checkpoint-4500"
)

model = AutoModelForCausalLM.from_pretrained(model_path).cuda().eval()
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [2]:
import torch

memory_size = 16
segment_size = 512
max_n_segments = 32
cell = MemoryCell(
    model,
    num_mem_tokens=memory_size,
)
model = RecurrentWrapper(
    cell,
    segment_size=segment_size,
    max_n_segments=max_n_segments,
    vary_n_segments=False,
    k2=-1,
)
model.eval()

memory = torch.load(f"{model_path}/memory.pt")
model.memory_cell.memory.data = memory

In [3]:
memory

Parameter containing:
tensor([[ 0.0068,  0.0035, -0.0195,  ..., -0.0270,  0.0347,  0.0532],
        [-0.0304,  0.0498, -0.0172,  ...,  0.0280, -0.0146,  0.0081],
        [-0.0309, -0.0164,  0.0071,  ..., -0.0090, -0.0518,  0.0320],
        ...,
        [ 0.0171,  0.0073,  0.0008,  ..., -0.0120,  0.0396, -0.0359],
        [ 0.0359, -0.0193,  0.0305,  ...,  0.0206, -0.0150,  0.0208],
        [ 0.0630, -0.0325, -0.0143,  ...,  0.0027,  0.0087, -0.0396]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)

In [4]:
model

RecurrentWrapper(
  (memory_cell): MemoryCell(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048, padding_idx=151643)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
              (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
              (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
              (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
              (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
            )
            (mlp): Qwen3MLP(
              (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
              (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
              (down_proj): Linear(in_features=6144, out_features=2048, bias

In [6]:
# prompt = "Give me a short introduction to large language model."
prompt = open("train_gym/distillation/sft/test_prompt.md").read()

prompt += "\n\nверни мне содержимое функции def offsets(self) в классе MemMapDataset целиком"
messages = [
    {
        "role": "user",
        "content": prompt,
    }
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer(
    [
        text,
    ],
    return_tensors="pt",
).to("cuda")
with torch.no_grad():
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    thinking_content = tokenizer.decode(
        output_ids[:index], skip_special_tokens=True
    ).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    print("thinking content:", thinking_content)
    print("content:", content)

thinking content: <think>

</think>
content: Вот содержимое функции `offsets` в классе `MemMapDataset`:

```python
def offsets(self):
    """
    Returns the offsets of the data in the dataset.

    Returns:
        list: A list of offsets for each data point in the dataset.
    """
    offsets = []
    for idx in range(self.n_samples):
        offset = idx * self.chunk_size
        offsets.append(offset)
    return offsets
```

Эта функция возвращает список, содержащий индексы каждого элемента в виде смещения в массиве данных. Каждый элемент списка соответствует смещению, которое нужно передать в `MemMapDataset` для чтения данных с диска.


In [7]:
text

'<|im_start|>user\n```python\nfrom __future__ import annotations\n\nfrom copy import deepcopy\nfrom typing import Any, Dict, List, Optional, Tuple, Type, Union\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom olmo.exceptions import OLMoEnvironmentError\n\nfrom ..aliases import PathOrStr\nfrom ..config import InstanceFilterConfig\nfrom ..util import _get_s3_client, file_size, get_bytes_range\nfrom .util import find_periodic_sequences, get_document_lengths\n\n__all__ = ["MemMapDataset"]\n\n\nclass MemMapDataset(Dataset[Dict[str, Any]]):\n    """\n    A PyTorch :class:`~torch.utils.data.Dataset` backed by one or more numpy memory-mapped arrays\n    of token IDs. Token IDs are chunked together into contiguous blocks of ``chunk_size``\n    to create instances.\n\n    If the length of a memory-mapped array is not a multiple of ``chunk_size`` the\n    remainder of the tokens will be ignored.\n\n    No special tokens are added to the input IDs so it\'s assumed 